In [1]:
from pyspark.sql.functions import *

sales_df = spark.read.table("sales_silver")

display(sales_df.limit(15))


StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b539a77-cd15-4a78-9d49-d3a937f2606f)

In [2]:
fact_sales = sales_df.select(
    "SalesOrderNumber",
    "SalesOrderLineNumber",
    "OrderDate",
    "CustomerName",
    "Item",
    "Quantity",
    "UnitPrice",
    "SalesAmount",
    "TaxAmount"
)

display(fact_sales)



StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 20008c00-220d-461f-bf45-a07bee6ad28c)

In [3]:
fact_sales.write.mode("overwrite").format("delta").saveAsTable("fact_sales")

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 5, Finished, Available, Finished, False)

In [4]:
product_summary = (
    sales_df
    .groupBy("Item")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        sum("SalesAmount").alias("TotalSales")
    )
)

display(product_summary)

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa721782-9752-4e7f-980c-892405b00d23)

In [5]:
product_summary.write.mode("overwrite").format("delta").saveAsTable("product_summary")

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 7, Finished, Available, Finished, False)

In [6]:
customer_summary = (
    sales_df
    .groupBy("CustomerName")
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("SalesAmount").alias("TotalRevenue")
    )
)

display(customer_summary)

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 83754a0a-9eac-4e45-8667-25694adc1359)

In [7]:
customer_summary.write.mode("overwrite").format("delta").saveAsTable("customer_summary")

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql.functions import *

date_dim = (
    sales_df
    .select("OrderDate")
    .distinct()
    .withColumn("Year", year("OrderDate"))
    .withColumn("Month", month("OrderDate"))
    .withColumn("MonthName", date_format("OrderDate", "MMMM"))
    .withColumn("Quarter", quarter("OrderDate"))
    .withColumn("YearMonth", date_format("OrderDate","yyyy-MM"))
)

display(date_dim)

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1f16e3d-1937-40d2-890f-2e8141b0c6d8)

In [9]:
# Overwrite the existing Delta table schema for `date_dim` to add the new `YearMonth` column
# This resolves the schema mismatch between the current table definition and the new DataFrame.
(
    date_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # allow schema changes (adds/removes columns)
    .saveAsTable("date_dim")
)

StatementMeta(, 58077c4c-d137-458b-a744-eac43b1648b0, 11, Finished, Available, Finished, False)